In [4]:
from ollama import generate, GenerateResponse

In [5]:
def Summarize(msg:str) -> str:
    prompt=f"""You are a financial text analysis expert. Your task is to analyze the provided article and generate a single summary sentence for sentiment analysis.

    **Instructions:**
    1. Read and understand the article thoroughly
    2. Identify the main financial topics, entities, and events discussed
    3. Generate ONE simple, clear sentence (15-25 words) that summarizes the article's core message
    4. Output ONLY the sentence - no labels, no explanations, no additional text

    **Article:**
    {msg}

    **Guidelines:**
    - Focus on financial terminology, company names, market movements, and economic indicators
    - The sentence must be grammatically correct and self-contained
    - Include clear sentiment indicators (positive, negative, or neutral words)
    - Avoid jargon; keep it accessible
    - Be objective and accurate to the source material

    **CRITICAL: Return ONLY the summary sentence. Do not include "Summary Sentence:" or any other labels or formatting.**"""

    response: GenerateResponse = generate(model="hf.co/us4/fin-llama3.1-8b:Q5_K_M", prompt=prompt)
    return response.response

In [ ]:
def Evaluate(sen:str) -> str:

    prompt=f"""You are a financial sentiment analysis expert. Your task is to evaluate the provided article and assign precise impact scores.

    **Instructions:**
    1. Analyze the sentiment and financial implications of the Article
    2. Identify ALL companies or entities mentioned in the article
    3. For EACH company, assign an impact score between -1.0 and 1.0 where:
       - -1.0 = extremely negative impact
       - -0.5 = moderately negative impact
       - 0.0 = neutral or no clear impact
       - 0.5 = moderately positive impact
       - 1.0 = extremely positive impact
    4. Consider market impact, investor sentiment, and economic implications for each company
    5. Use decimal precision (e.g., -0.73, 0.42, 0.15)

    **Article to Analyze:**
    {sen}

    **Scoring Guidelines:**
    - Strong negative words (crash, plunge, collapse, crisis) → -0.7 to -1.0
    - Moderate negative words (decline, fall, weak, concern) → -0.3 to -0.6
    - Neutral words (stable, unchanged, maintained) → -0.2 to 0.2
    - Moderate positive words (rise, growth, improvement, gain) → 0.3 to 0.6
    - Strong positive words (surge, soar, breakthrough, boom) → 0.7 to 1.0

    **Output Format:**
    Return ONLY a single float number between -1.0 and 1.0
    
    **Examples of valid outputs:**
    0.75
    -0.42
    0.0
    -0.88
    0.33

    **CRITICAL RULES:**
    - Output must be a valid float number with up to 2 decimal places
    - No explanations, no text, no labels, no markdown
    - No words like "score:", "impact:", or any prefixes
    - Just the number itself (e.g., 0.65)""" 


    response: GenerateResponse = generate(model="hf.co/us4/fin-llama3.1-8b:Q5_K_M", 
    prompt=prompt,
    options={
    'temperature': 0.1,
    'top_p': 0.9,
    'top_k': 40,
    'num_predict': 300,
    'repeat_penalty': 1.0,
    'num_ctx': 2048,
    'num_thread': 16,
    })

    validated_list = []

    for token in json.loads(response.response):
        print(token)
        validated_string = fuzzy_match(str(token).replace("'", '"'))
        if validated_string:
            validated_list.append(validated_string)

    return str(validated_list)

In [16]:
msg = r"Vedanta poised for 16% annual growth in pre-tax earnings through FY28 on volume ramp-up"

summary = Summarize(msg)
print(summary)

eval = Evaluate(summary)
print(eval)
print(type(eval))

Vedanta's stock is poised for 16% annual growth in pre-tax earnings through FY28 due to strong volume ramp-up strategies.
{'company': 'Vedanta', 'impact_score': 0.5}
['{"company": "Vedanta Limited", "impact_score": 0.5}']
<class 'str'>


In [13]:
from thefuzz import fuzz
import pandas as pd
import json

def fuzzy_match(LLM_json_output: str) -> str|None: 
    token = json.loads(LLM_json_output)
    name = token['company']
    impact_score = token['impact_score']


    with open('StockNames.csv', 'r') as f:
        reader = pd.read_csv(f)
        company_list = reader['Names'].tolist()


    for company in company_list:
        ratio_partial = fuzz.partial_ratio(name, company)
        if ratio_partial > 92:
            return f'{{"company": "{company}", "impact_score": {impact_score}}}'

            
        
        ratio_full = fuzz.ratio(name, company)
        if ratio_full > 92:
            return company

        ratio_token = fuzz.token_sort_ratio(name, company)
        if ratio_token > 92:
            return company
        
    return None

In [1]:
%pip install yfinance

  Using cached multitasking-0.0.12.tar.gz (19 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached peewee-3.18.3-py3-none-any.whl
  Using cached beautifulsoup4-4.14.2-py3-none-any.whl.metadata (3.8 kB)
  Using cached curl_cffi-0.13.0-cp39-abi3-win_amd64.whl.metadata (13 kB)
Using cached beautifulsoup4-4.14.2-py3-none-any.whl (106 kB)
Using cached curl_cffi-0.13.0-cp39-abi3-win_amd64.whl (1.6 MB)
  Created wheel for multitasking: filename=multitasking-0.0.12-py3-none-any.whl size=15703 sha256=5a97d0ae7adaab0e53d386847685ea4f14f0e056a3eeac83b771458e88742739
  Stored in directory: c:\users\hp\appdata\local\pip\cache\wheels\e9\25\85\25d2e1cfc0ece64b930b16972f7e4cc3599c43b531f1eba06d
Successfully bui

In [1]:
pip install nsetools


   ---------------------------------------- 0/2 [dateutils]
   ---------------------------------------- 0/2 [dateutils]
   ---------------------------------------- 0/2 [dateutils]
   ---------------------------------------- 0/2 [dateutils]
   -------------------- ------------------- 1/2 [nsetools]
   ---------------------------------------- 2/2 [nsetools]

Note: you may need to restart the kernel to use updated packages.


In [1]:
pip install pymongo

   ---------------------------------------- 0.0/808.1 kB ? eta -:--:--
   ------------ --------------------------- 262.1/808.1 kB ? eta -:--:--
   ------------------------- -------------- 524.3/808.1 kB 2.4 MB/s eta 0:00:01
   ---------------------------------------- 808.1/808.1 kB 2.1 MB/s  0:00:00

   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   ---------------------------------------- 0/2 [dnspython]
   -----------------------------------

In [11]:
import yfinance as yf

a = yf.Ticker('20MICRONS.NS').get_news()[0]['content']['title']
print(type(a))

<class 'str'>


In [12]:
import json
import yfinance as yf
import requests
import datetime
from pymongo import MongoClient

client = MongoClient('mongodb://localhost:27017/')
db = client['Finnalyze']
collection = db['stockData']


with open('stockList.json', 'r') as f:
    stockList = json.load(f)

while True:
    try:
        for stock in stockList:
            stockCode = stock['code'] + '.NS'
            news = yf.Ticker(stockCode).get_news()
        
            for article in news:
                content = article['content']['summary']
                title = article['content']['title']
                print(content)
            
                response = requests.post(
                    'http://localhost:5000/evaluate',
                    json={"article": content}
                    )

                impact_score = float(response.json())
                print(type(impact_score))
                print(impact_score)

                updates = {
                    "prediction" : 'down' if impact_score < 0 else 'up',
                    "latest_headline": title,
                    "latest_article": content,
                    "updated_at": datetime.datetime.utcnow()
                }
    
                result = collection.update_one(
                    {"code": stockCode.replace('.NS', '')},
                    {"$set": updates}
                )
                
                if result.matched_count > 0:
                    print(f"✓ Updated stock: {stockCode}")
                else:
                    print(f"✗ Stock not found: {stockCode}")
    
    except KeyboardInterrupt as e:
        break

    except Exception as e:
        print(e)
        continue
        


Despite revenue dips, 20 Microns Ltd (BOM:533022) showcases strong EBITDA growth and cost management, with a positive outlook for the second half of the year.
<class 'float'>
0.53
✓ Updated stock: 20MICRONS.NS
(Reuters) -Diversified products maker 3M India reported a 43% rise in second-quarter profit on Monday, led by strong demand across all its segments.
<class 'float'>
0.65
✓ Updated stock: 3MINDIA.NS
The company, which makes everything from 'Post-it' notes to medical gear, reported a 13.1% rise in profit after tax to 1.78 billion rupees ($20.30 million) for the first quarter ended June 30, from 1.57 billion rupees a year ago.  The Indian arm of U.S. conglomerate 3M has logged robust demand in sectors such as healthcare, infrastructure and electronics as India's own manufacturing activities scale up, led by higher government expenditure and strong economic growth.
<class 'float'>
0.53
✓ Updated stock: 3MINDIA.NS
Despite a decline in new customer acquisition, 5paisa Capital Ltd (NSE:

In [ ]:
import yfinance as yf

def get_stock_news_links(ticker_symbol):
    """
    Fetches the latest news titles and links for a given stock ticker.
"""
        # Create a Ticker object
    ticker = yf.Ticker(ticker_symbol)

        # Fetch news items
    news_items = ticker.news
        
    if not news_items:
        print(f"No news found for {ticker_symbol}")
        return

    print(f"--- Latest News for {ticker_symbol} ---")
    for i, article in enumerate(news_items):
        title = article.get('title')

        if title:
            print(f"* Title: {title}")
            print("-" * 20)

# Example usage for Apple Inc. (AAPL)
get_stock_news_links("")

--- Latest News for AAPL ---
